In [ ]:
import numpy as np

# Create a 2-D array, set every second element in
# some rows and find max per row:

x = np.arange(15, dtype=np.int64).reshape(3, 5)
x[1:, ::2] = -99
x
# array([[  0,   1,   2,   3,   4],
#        [-99,   6, -99,   8, -99],
#        [-99,  11, -99,  13, -99]])

x.max(axis=1)
# array([ 4,  8, 13])

# Generate normally distributed random numbers:
rng = np.random.default_rng()
samples = rng.normal(size=2500)
samples

# Add another NumPy array, multiply element-wise with x, and print results:
y = np.arange(1, 16, dtype=np.int64).reshape(3, 5)
product = x * y
print("Second array (y):")
print(y)
print("Element-wise multiplication result (x * y):")
print(product)



# Compute the diameter (max - min) of the new array and multiply it by 2:
diameter = np.ptp(product)
print("Diameter of product array:", diameter)
print("Diameter multiplied by 2:", diameter * 2)

# Add an array with even numbers and multiply it with the newly produced array:
even_numbers = np.arange(2, 32, 2, dtype=np.int64).reshape(3, 5)
final_product = product * even_numbers
print("Even numbers array:")
print(even_numbers)
print("Multiplication of product with even_numbers:")
print(final_product)

Second array (y):
[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]]
Element-wise multiplication result (x * y):
[[    0     2     6    12    20]
 [ -594    42  -792    72  -990]
 [-1089   132 -1287   182 -1485]]
Diameter of product array: 1667
Diameter multiplied by 2: 3334
Even numbers array:
[[ 2  4  6  8 10]
 [12 14 16 18 20]
 [22 24 26 28 30]]
Multiplication of product with even_numbers:
[[     0      8     36     96    200]
 [ -7128    588 -12672   1296 -19800]
 [-23958   3168 -33462   5096 -44550]]


In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize()

In [10]:
map = geemap.Map()
map.add_basemap("HYBRID")
map.set_center(-122.4194, 37.7749, 10)
map

Map(center=[37.7749, -122.4194], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

In [ ]:
roi = map.draw_last_feature.geometry()
print("ROI geometry:", roi)

In [22]:
# Creating a timelist
time_start = ee.Date('2005')
time_end = ee.Date('2025')
time_dif = time_end.difference(time_start, 'year').round()
time_list = ee.List.sequence(0, ee.Number(time_dif).subtract(1)).map(
    lambda x: time_start.advance(x, 'year')
)

# Function for obtaining the date for the spicfic time
def annual(date, col):
  start_date = ee.Date(date)
  end_date = start_date.advance(1, 'year')
  img_sum = col.filterDate(start_date, end_date).sum()
  return img_sum.set('system:time_start', start_date.millis())

In [25]:
vis_params4 = {'bands': ['NDVI'], 'palette': ['#f7fcf5', '#c7e9c0', '#74c476', '#238b45', '#00441b'],
              'min': -7970.3, 'max': 87319.55}

In [23]:
# Script in the cell is Demo
ndvi = (
    ee.ImageCollection("MODIS/061/MOD13A2")
    .filterDate(time_start, time_end)
    .select("NDVI")
    .map(lambda x: x.multiply(0.0001).copyProperties(x, ['system:time_start'])
    )
)

ndvi_annual = ee.ImageCollection(time_list.map(lambda x: annual(date=x, col=ndvi)))

In [26]:
map.addLayer(ndvi.mean().clip(roi), vis_params4, "NDVI")